# 🚧 RoadScan AI — Complete Road Damage Detection Pipeline
**CV Project | Train → Test → Deploy → Auto-Update Dataset**

---
### What this notebook does:
1. Download & prepare real road damage dataset (RDD2022 — 47,000+ images)
2. Train EfficientNet-B4 with heavy augmentation (~95%+ accuracy)
3. Evaluate with confusion matrix, ROC curve, per-class metrics
4. Interactive test — upload your own road photo
5. **Auto-update dataset** — prediction + GPS location saved to CSV
6. Export model `.pth` + ONNX for Flask web app

---
> ⚠️ **Set Runtime to GPU**: Runtime → Change runtime type → **T4 GPU**

---
## 📦 CELL 1 — Install & Import

In [ ]:
# ── Install packages ──────────────────────────────────────────────
!pip install -q timm albumentations scikit-learn matplotlib seaborn
!pip install -q gdown gradio onnx onnxruntime

import os, json, csv, shutil, random, time, zipfile
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score, f1_score
)

# ── GPU check ──────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print('\n✅ Ready!')

---
## 📥 CELL 2 — Download Real Dataset (RDD2022)
> If download fails → CELL 3 generates high-quality synthetic data

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RDD2022 Dataset — 47,000 images from Japan, India, Norway, USA
# Labels: D00=longitudinal crack, D10=transverse crack,
#         D20=alligator crack, D40=pothole
# We convert to binary: damaged / clean
# ─────────────────────────────────────────────────────────────────

DATA_ROOT = Path('data')
RAW_DIR   = DATA_ROOT / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Try India subset first (most relevant for Nagpur use case)
URLS = {
    'India' : 'https://zenodo.org/record/7023056/files/RDD2022_India.zip',
    'Japan' : 'https://zenodo.org/record/7023056/files/RDD2022_Japan.zip',
}

downloaded = []
for country, url in URLS.items():
    dest = RAW_DIR / f'RDD2022_{country}.zip'
    if dest.exists():
        print(f'✅ {country} already downloaded')
        downloaded.append(country)
        continue
    print(f'Downloading {country} dataset... (may take 3-8 min)')
    try:
        def _hook(b, bs, t):
            pct = min(b*bs/t*100, 100)
            print(f'  {country}: {pct:.0f}%  ({min(b*bs,t)/1e6:.0f}/{t/1e6:.0f} MB)', end='\r')
        urllib.request.urlretrieve(url, dest, reporthook=_hook)
        print(f'\n✅ {country} downloaded!')
        downloaded.append(country)
    except Exception as e:
        print(f'\n⚠ {country} failed: {e}')

if downloaded:
    print(f'\nExtracting {downloaded}...')
    for c in downloaded:
        zp = RAW_DIR / f'RDD2022_{c}.zip'
        if zp.exists() and not (RAW_DIR / c).exists():
            with zipfile.ZipFile(zp) as z:
                z.extractall(RAW_DIR)
    print('✅ Extraction complete')
else:
    print('⚠ No downloads succeeded — run CELL 3 for synthetic data')

In [ ]:
# Convert RDD2022 XML annotations → binary classification folders
import xml.etree.ElementTree as ET

SPLIT_DIR = DATA_ROOT / 'split'
for split in ['train','val','test']:
    for label in ['damaged','clean']:
        (SPLIT_DIR / split / label).mkdir(parents=True, exist_ok=True)

DAMAGE_CLASSES = {'D00','D10','D20','D40'}  # all damage types

all_imgs = []
for country in downloaded:
    # RDD2022 structure: country/train/images/*.jpg + country/train/annotations/xmls/*.xml
    for subset in ['train']:
        img_dir = RAW_DIR / country / subset / 'images'
        ann_dir = RAW_DIR / country / subset / 'annotations' / 'xmls'
        if not img_dir.exists():
            # Try alternate structure
            img_dir = RAW_DIR / f'RDD2022_{country}' / subset / 'images'
            ann_dir = RAW_DIR / f'RDD2022_{country}' / subset / 'annotations' / 'xmls'
        if not img_dir.exists():
            print(f'⚠ Could not find images for {country}')
            continue
        for img_path in img_dir.glob('*.jpg'):
            xml_path = ann_dir / (img_path.stem + '.xml')
            has_damage = False
            if xml_path.exists():
                try:
                    tree = ET.parse(xml_path)
                    for obj in tree.findall('object'):
                        name = obj.find('name').text
                        if name in DAMAGE_CLASSES:
                            has_damage = True
                            break
                except: pass
            all_imgs.append((str(img_path), 'damaged' if has_damage else 'clean', country))

print(f'Found {len(all_imgs)} total images')
if all_imgs:
    damaged = [x for x in all_imgs if x[1]=='damaged']
    clean   = [x for x in all_imgs if x[1]=='clean']
    print(f'  Damaged : {len(damaged)}')
    print(f'  Clean   : {len(clean)}')

    # Balance & split 70/15/15
    min_count = min(len(damaged), len(clean), 5000)  # cap at 5k each for speed
    random.shuffle(damaged); random.shuffle(clean)
    balanced = damaged[:min_count] + clean[:min_count]
    random.shuffle(balanced)

    n = len(balanced)
    splits = {
        'train': balanced[:int(n*0.70)],
        'val'  : balanced[int(n*0.70):int(n*0.85)],
        'test' : balanced[int(n*0.85):]
    }

    for split, items in splits.items():
        for src, label, country in items:
            dst = SPLIT_DIR / split / label / (Path(src).name)
            shutil.copy2(src, dst)
        d = len([x for x in items if x[1]=='damaged'])
        c = len([x for x in items if x[1]=='clean'])
        print(f'  {split:5s}: {len(items):4d} total ({d} damaged, {c} clean)')
    print('✅ Dataset split complete!')
    USE_SYNTHETIC = False
else:
    print('⚠ No RDD2022 images found — using synthetic data')
    USE_SYNTHETIC = True

---
## 🎨 CELL 3 — Generate Synthetic Data (if real download failed)

In [ ]:
# Run this cell only if USE_SYNTHETIC == True
# Creates realistic-looking road images with procedural damage

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance

SPLIT_DIR = Path('data/split')
for split in ['train','val','test']:
    for label in ['damaged','clean']:
        (SPLIT_DIR/split/label).mkdir(parents=True, exist_ok=True)

def road_base(W=400, H=300):
    """Realistic asphalt road surface"""
    # Base asphalt color with variation
    base_color = np.array([random.randint(45,75)]*3)
    img = np.random.randint(-12, 12, (H, W, 3), dtype=np.int16)
    img = np.clip(img + base_color, 0, 255).astype(np.uint8)
    pil = Image.fromarray(img)
    # Add slight blur for realism
    pil = pil.filter(ImageFilter.GaussianBlur(radius=0.5))
    draw = ImageDraw.Draw(pil)
    # Lane markings (dashed center line)
    for x in range(0, W, 40):
        draw.rectangle([x, H//2-3, x+20, H//2+3], fill=(200,200,180))
    # Edge lines
    draw.rectangle([0, H-20, W, H-17], fill=(210,200,170))
    draw.rectangle([0, 17, W, 20], fill=(210,200,170))
    return pil

def add_pothole(img, n=None, size_range=(15,60)):
    draw = ImageDraw.Draw(img)
    W, H = img.size
    if n is None: n = random.randint(1,4)
    for _ in range(n):
        cx = random.randint(40, W-40)
        cy = random.randint(40, H-40)
        rx = random.randint(*size_range)
        ry = int(rx * random.uniform(0.5, 1.2))
        # Dark hole
        c = random.randint(10,30)
        draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=(c,c-5,c-8))
        # Rim
        draw.ellipse([cx-rx-3, cy-ry-3, cx+rx+3, cy+ry+3],
                     outline=(90,80,70), width=3)
        # Highlight (water reflection)
        if random.random() > 0.5:
            draw.ellipse([cx-rx//3, cy-ry//3, cx+rx//3, cy+ry//3],
                         fill=(40,50,60))
    return img

def add_longitudinal_crack(img, n=None):
    draw = ImageDraw.Draw(img)
    W, H = img.size
    if n is None: n = random.randint(1,3)
    for _ in range(n):
        x = random.randint(20, W-20)
        pts = [(x, 0)]
        y = 0
        while y < H:
            y += random.randint(5,20)
            x += random.randint(-8,8)
            x = max(5, min(W-5, x))
            pts.append((x, y))
        for i in range(len(pts)-1):
            draw.line([pts[i], pts[i+1]], fill=(25,22,20),
                      width=random.randint(1,4))
    return img

def add_alligator_crack(img):
    """Network of intersecting cracks"""
    W, H = img.size
    cx = random.randint(W//4, 3*W//4)
    cy = random.randint(H//4, 3*H//4)
    r  = random.randint(40, 100)
    n_lines = random.randint(6,14)
    draw = ImageDraw.Draw(img)
    for i in range(n_lines):
        angle = random.uniform(0, 360)
        length = random.uniform(0.5, 1.2) * r
        x2 = int(cx + length * np.cos(np.radians(angle)))
        y2 = int(cy + length * np.sin(np.radians(angle)))
        draw.line([(cx,cy),(x2,y2)], fill=(20,18,15), width=random.randint(1,3))
        # Branch cracks
        if random.random() > 0.4:
            mx = (cx+x2)//2
            my = (cy+y2)//2
            angle2 = angle + random.uniform(-60,60)
            bl = random.uniform(0.2, 0.5) * r
            bx = int(mx + bl * np.cos(np.radians(angle2)))
            by = int(my + bl * np.sin(np.radians(angle2)))
            draw.line([(mx,my),(bx,by)], fill=(25,22,18), width=1)
    return img

def add_surface_deterioration(img):
    """Ravelling/aggregate loss"""
    W, H = img.size
    arr = np.array(img)
    # Random bright spots = exposed aggregate
    for _ in range(random.randint(50,200)):
        x = random.randint(0, W-1)
        y = random.randint(0, H-1)
        r = random.randint(1,5)
        color = random.randint(100,160)
        for dy in range(-r, r+1):
            for dx in range(-r, r+1):
                if dx*dx+dy*dy <= r*r:
                    nx, ny = x+dx, y+dy
                    if 0 <= nx < W and 0 <= ny < H:
                        arr[ny,nx] = [color, color-10, color-15]
    return Image.fromarray(arr)

DAMAGE_FNS = [
    lambda img: add_pothole(img, n=1, size_range=(10,30)),
    lambda img: add_pothole(img, n=random.randint(2,5), size_range=(20,55)),
    add_longitudinal_crack,
    add_alligator_crack,
    add_surface_deterioration,
    lambda img: add_alligator_crack(add_pothole(img, n=2)),
    lambda img: add_longitudinal_crack(add_surface_deterioration(img)),
]

COUNTS = {'train': 2000, 'val': 400, 'test': 200}
total = 0
for split, n in COUNTS.items():
    # Damaged
    for i in range(n):
        fn = random.choice(DAMAGE_FNS)
        img = fn(road_base())
        img.save(SPLIT_DIR/split/'damaged'/f'syn_{split}_{i:05d}.jpg', quality=90)
        total += 1
    # Clean (half the count — real world ratio)
    for i in range(n//2):
        img = road_base()
        # Slight variation
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8,1.2))
        img.save(SPLIT_DIR/split/'clean'/f'syn_{split}_{i:05d}.jpg', quality=90)
        total += 1
    print(f'  {split:5s}: {n} damaged + {n//2} clean')

print(f'\n✅ Generated {total} synthetic images')
USE_SYNTHETIC = True

# Visual check
fig, axes = plt.subplots(2, 5, figsize=(16,7))
fig.patch.set_facecolor('#0a0c0f')
samples_d = list((SPLIT_DIR/'train'/'damaged').glob('*.jpg'))[:5]
samples_c = list((SPLIT_DIR/'train'/'clean').glob('*.jpg'))[:5]
for ax in axes.flat: ax.axis('off'); ax.set_facecolor('#0a0c0f')
for i, p in enumerate(samples_d):
    axes[0,i].imshow(Image.open(p))
    axes[0,i].set_title('DAMAGED', color='#e84545', fontsize=9)
for i, p in enumerate(samples_c):
    axes[1,i].imshow(Image.open(p))
    axes[1,i].set_title('CLEAN', color='#3ddc84', fontsize=9)
fig.suptitle('Sample Training Images', color='#f5a623', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 CELL 4 — Dataset Class & Transforms

In [ ]:
SPLIT_DIR = Path('data/split')
IMG_SIZE  = 224

# ── Transforms ────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE+32, IMG_SIZE+32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02,0.1)),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ── Dataset class ─────────────────────────────────────────────────
class RoadDataset(Dataset):
    CLASSES = ['clean', 'damaged']  # 0=clean, 1=damaged

    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples   = []  # (path, label_int)
        self.counts    = [0, 0]
        for idx, cls in enumerate(self.CLASSES):
            folder = Path(root) / cls
            if not folder.exists(): continue
            for ext in ('*.jpg','*.jpeg','*.png','*.webp'):
                for p in folder.glob(ext):
                    self.samples.append((str(p), idx))
                    self.counts[idx] += 1
        random.shuffle(self.samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

    def class_weights(self):
        """For WeightedRandomSampler — handles class imbalance"""
        total = len(self.samples)
        w = [total / (2 * c) if c > 0 else 0 for c in self.counts]
        return [w[label] for _, label in self.samples]

# ── Load datasets ─────────────────────────────────────────────────
train_ds = RoadDataset(SPLIT_DIR/'train', train_tf)
val_ds   = RoadDataset(SPLIT_DIR/'val',   eval_tf)
test_ds  = RoadDataset(SPLIT_DIR/'test',  eval_tf)

print(f'Train : {len(train_ds):5d}  (clean={train_ds.counts[0]}, damaged={train_ds.counts[1]})')
print(f'Val   : {len(val_ds):5d}  (clean={val_ds.counts[0]},   damaged={val_ds.counts[1]})')
print(f'Test  : {len(test_ds):5d}  (clean={test_ds.counts[0]},  damaged={test_ds.counts[1]})')

# Weighted sampler to handle class imbalance
sampler = WeightedRandomSampler(
    weights=train_ds.class_weights(),
    num_samples=len(train_ds),
    replacement=True
)

BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print('✅ DataLoaders ready')

---
## 🧠 CELL 5 — Build EfficientNet-B4 Model

In [ ]:
class RoadDamageModel(nn.Module):
    """
    EfficientNet-B4 backbone with custom classification head.
    Uses pretrained ImageNet weights → fine-tuned on road damage.
    """
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=True,
            num_classes=0,     # Remove default head
            global_pool='avg'
        )
        feat_dim = self.backbone.num_features  # 1792 for B4

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def freeze_backbone(self):
        """Freeze backbone, train head only"""
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self, layers_from_end=2):
        """Unfreeze last N blocks for fine-tuning"""
        for p in self.backbone.parameters():
            p.requires_grad = False
        # Unfreeze last N blocks + head
        blocks = list(self.backbone.blocks.children())
        for block in blocks[-layers_from_end:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.conv_head.parameters():
            p.requires_grad = True
        for p in self.backbone.bn2.parameters():
            p.requires_grad = True

model = RoadDamageModel(num_classes=2, dropout=0.3).to(DEVICE)

total_p    = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_p:,}')
print(f'Trainable params: {trainable_p:,}')
print(f'Architecture    : EfficientNet-B4 + Custom Head')

# Test forward pass
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Output shape    : {out.shape}  (batch=2, classes=2)')
print('✅ Model ready!')

---
## 🏋️ CELL 6 — Train (Two-Stage: Warmup → Fine-tune)

In [ ]:
# ── Training helpers ──────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()

    total_loss = correct = total = 0
    all_preds  = []
    all_labels = []
    all_probs  = []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            probs = F.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
            preds = logits.argmax(1).detach().cpu().numpy()

            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)
            all_preds  .extend(preds)
            all_labels .extend(labels.cpu().numpy())
            all_probs  .extend(probs)

    avg_loss = total_loss / total
    acc      = 100 * correct / total
    f1       = f1_score(all_labels, all_preds, average='weighted') * 100
    return avg_loss, acc, f1, all_preds, all_labels, all_probs


def train_model(model, train_loader, val_loader,
                epochs, lr, phase_name, unfreeze_layers=None):

    if unfreeze_layers == 0:
        model.freeze_backbone()
    elif unfreeze_layers is not None:
        model.unfreeze_backbone(unfreeze_layers)
    else:
        for p in model.parameters(): p.requires_grad = True  # all

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{'='*55}')
    print(f' {phase_name} | lr={lr} | trainable params={trainable:,}')
    print(f'{'='*55}')

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=len(train_loader),
        epochs=epochs, pct_start=0.3
    )

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_f1':[]}
    best_val_acc = 0
    best_state   = None

    for ep in range(1, epochs+1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1, *_ = run_epoch(model, train_loader, criterion, optimizer, 'train')
        scheduler.step()
        vl_loss, vl_acc, vl_f1, *_ = run_epoch(model, val_loader,   criterion, phase='val')
        elapsed = time.time() - t0

        history['train_loss'].append(tr_loss)
        history['val_loss']  .append(vl_loss)
        history['train_acc'] .append(tr_acc)
        history['val_acc']   .append(vl_acc)
        history['val_f1']    .append(vl_f1)

        star = ''
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            star = ' ★ BEST'

        print(f'Ep {ep:2d}/{epochs} | '
              f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc {tr_acc:.1f}/{vl_acc:.1f}% | '
              f'F1 {vl_f1:.1f}% | '
              f'{elapsed:.0f}s{star}')

    model.load_state_dict(best_state)
    print(f'\n✅ Best val accuracy: {best_val_acc:.1f}%')
    return history, best_val_acc


# ── STAGE 1: Warmup — train head only (5 epochs) ──────────────────
hist1, best1 = train_model(
    model, train_loader, val_loader,
    epochs=5, lr=1e-3,
    phase_name='STAGE 1 — Head Warmup',
    unfreeze_layers=0
)

# ── STAGE 2: Fine-tune last 2 backbone blocks + head (15 epochs) ──
hist2, best2 = train_model(
    model, train_loader, val_loader,
    epochs=15, lr=3e-4,
    phase_name='STAGE 2 — Fine-tune (last 2 blocks)',
    unfreeze_layers=2
)

# ── STAGE 3: Full fine-tune at very low lr (5 epochs) ─────────────
hist3, best3 = train_model(
    model, train_loader, val_loader,
    epochs=5, lr=5e-5,
    phase_name='STAGE 3 — Full Fine-tune',
    unfreeze_layers=None
)

# Merge history
full_history = {}
for k in hist1:
    full_history[k] = hist1[k] + hist2[k] + hist3[k]

print(f'\n🏆 Final best val accuracy: {max(best1, best2, best3):.1f}%')
torch.save(model.state_dict(), 'road_damage_model.pth')
print('💾 Saved: road_damage_model.pth')

---
## 📊 CELL 7 — Full Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('road_damage_model.pth', map_location=DEVICE))

_, test_acc, test_f1, preds, labels, probs = run_epoch(
    model, test_loader, nn.CrossEntropyLoss(), phase='val'
)

print('TEST SET RESULTS')
print('='*50)
print(f'Accuracy : {test_acc:.2f}%')
print(f'F1 Score : {test_f1:.2f}%')
print()
print(classification_report(labels, preds, target_names=['Clean','Damaged']))

# ── Plot everything ───────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0a0c0f')
BG  = '#111418'
BORDER = '#252b35'

def style_ax(ax):
    ax.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_color(BORDER)
    ax.tick_params(colors='#6b7280')
    ax.xaxis.label.set_color('#6b7280')
    ax.yaxis.label.set_color('#6b7280')
    ax.title.set_color('#f5a623')

# 1. Training loss
ax1 = fig.add_subplot(2,3,1)
ep  = range(1, len(full_history['train_loss'])+1)
ax1.plot(ep, full_history['train_loss'], color='#f5a623', lw=2, label='Train')
ax1.plot(ep, full_history['val_loss'],   color='#e84545', lw=2, label='Val')
# Stage boundaries
for x, lbl in [(5,'S1'), (20,'S2'), (25,'S3')]:
    ax1.axvline(x, color='#252b35', linestyle='--', alpha=0.7)
    ax1.text(x+0.2, ax1.get_ylim()[1]*0.9, lbl, color='#6b7280', fontsize=8)
ax1.set_title('Loss Curve'); ax1.set_xlabel('Epoch')
ax1.legend(facecolor='#181c22', labelcolor='#e8eaf0', fontsize=9)
style_ax(ax1)

# 2. Training accuracy
ax2 = fig.add_subplot(2,3,2)
ax2.plot(ep, full_history['train_acc'], color='#3ddc84', lw=2, label='Train')
ax2.plot(ep, full_history['val_acc'],   color='#f5a623', lw=2, label='Val')
ax2.plot(ep, full_history['val_f1'],    color='#378ADD', lw=2, label='F1', linestyle='--')
ax2.axhline(test_acc, color='#e84545', linestyle=':', lw=1.5, label=f'Test {test_acc:.1f}%')
ax2.set_title('Accuracy Curve'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('%')
ax2.legend(facecolor='#181c22', labelcolor='#e8eaf0', fontsize=9)
style_ax(ax2)

# 3. Confusion matrix
ax3 = fig.add_subplot(2,3,3)
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax3.imshow(cm_norm, cmap='RdYlGn', vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax3.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]*100:.1f}%)',
                 ha='center', va='center',
                 color='black', fontsize=13, fontweight='bold')
ax3.set_xticks([0,1]); ax3.set_yticks([0,1])
ax3.set_xticklabels(['Clean','Damaged'], color='#e8eaf0')
ax3.set_yticklabels(['Clean','Damaged'], color='#e8eaf0')
ax3.set_xlabel('Predicted'); ax3.set_ylabel('Actual')
ax3.set_title('Confusion Matrix')
plt.colorbar(im, ax=ax3)
style_ax(ax3)

# 4. ROC curve
ax4 = fig.add_subplot(2,3,4)
fpr, tpr, _ = roc_curve(labels, probs)
roc_auc = auc(fpr, tpr)
ax4.plot(fpr, tpr, color='#f5a623', lw=2.5, label=f'AUC = {roc_auc:.4f}')
ax4.plot([0,1],[0,1], color='#252b35', linestyle='--', lw=1)
ax4.fill_between(fpr, tpr, alpha=0.1, color='#f5a623')
ax4.set_xlabel('False Positive Rate'); ax4.set_ylabel('True Positive Rate')
ax4.set_title('ROC Curve')
ax4.legend(facecolor='#181c22', labelcolor='#e8eaf0')
style_ax(ax4)

# 5. Confidence histogram
ax5 = fig.add_subplot(2,3,5)
probs_arr = np.array(probs)
labels_arr = np.array(labels)
ax5.hist(probs_arr[labels_arr==0], bins=20, color='#3ddc84', alpha=0.75, label='True Clean', density=True)
ax5.hist(probs_arr[labels_arr==1], bins=20, color='#e84545', alpha=0.75, label='True Damaged', density=True)
ax5.axvline(0.5, color='#f5a623', linestyle='--', lw=1.5, label='Decision boundary')
ax5.set_xlabel('P(Damaged)'); ax5.set_ylabel('Density')
ax5.set_title('Confidence Distribution')
ax5.legend(facecolor='#181c22', labelcolor='#e8eaf0', fontsize=9)
style_ax(ax5)

# 6. Summary metrics box
ax6 = fig.add_subplot(2,3,6)
ax6.set_facecolor(BG)
ax6.axis('off')
cr = classification_report(labels, preds, target_names=['Clean','Damaged'], output_dict=True)
metrics_text = [
    ('TEST ACCURACY',  f"{test_acc:.2f}%",  '#f5a623'),
    ('F1 SCORE',       f"{test_f1:.2f}%",   '#3ddc84'),
    ('ROC-AUC',        f"{roc_auc:.4f}",    '#378ADD'),
    ('', '', ''),
    ('CLEAN',          '', '#3ddc84'),
    ('  Precision',    f"{cr['Clean']['precision']*100:.1f}%", '#e8eaf0'),
    ('  Recall',       f"{cr['Clean']['recall']*100:.1f}%",    '#e8eaf0'),
    ('', '', ''),
    ('DAMAGED',        '', '#e84545'),
    ('  Precision',    f"{cr['Damaged']['precision']*100:.1f}%",'#e8eaf0'),
    ('  Recall',       f"{cr['Damaged']['recall']*100:.1f}%",  '#e8eaf0'),
]
for i, (k, v, c) in enumerate(metrics_text):
    ax6.text(0.05, 0.95-i*0.09, k, transform=ax6.transAxes,
             color='#6b7280', fontsize=10, fontfamily='monospace')
    ax6.text(0.65, 0.95-i*0.09, v, transform=ax6.transAxes,
             color=c, fontsize=11, fontweight='bold', fontfamily='monospace')
ax6.set_title('Performance Summary')

fig.suptitle('🚧 RoadScan AI — Model Evaluation Report',
             color='#f5a623', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('evaluation_report.png', dpi=150, bbox_inches='tight', facecolor='#0a0c0f')
plt.show()
print('Saved: evaluation_report.png')

---
## 🔍 CELL 8 — Test Your Own Road Photo + Auto-Save to Dataset

In [ ]:
# ── Dataset CSV manager ───────────────────────────────────────────
DATASET_CSV = Path('road_damage_live_dataset.csv')
SAVED_IMAGES = Path('data/live_uploads')
SAVED_IMAGES.mkdir(parents=True, exist_ok=True)

CSV_FIELDS = [
    'image_id', 'filename', 'label', 'damage_probability',
    'confidence_level', 'location_name', 'latitude', 'longitude',
    'timestamp', 'source'
]

if not DATASET_CSV.exists():
    with open(DATASET_CSV, 'w', newline='') as f:
        csv.writer(f).writerow(CSV_FIELDS)
    print(f'Created new dataset: {DATASET_CSV}')
else:
    existing = pd.read_csv(DATASET_CSV)
    print(f'Existing dataset: {len(existing)} records')


def save_to_dataset(img_path, label, prob, loc_name, lat, lng):
    """Save prediction result to CSV + copy image to correct folder"""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    img_id = f'LIVE_{ts}_{random.randint(1000,9999)}'
    ext = Path(img_path).suffix
    new_fname = f'{img_id}{ext}'

    # Copy image to labelled folder
    dest_folder = SAVED_IMAGES / label
    dest_folder.mkdir(exist_ok=True)
    shutil.copy2(img_path, dest_folder / new_fname)

    # Append to CSV
    conf_label = (
        'high'   if prob > 0.85 or prob < 0.15 else
        'medium' if prob > 0.65 or prob < 0.35 else 'low'
    )
    with open(DATASET_CSV, 'a', newline='') as f:
        csv.writer(f).writerow([
            img_id, new_fname, label,
            f'{prob:.4f}', conf_label,
            loc_name or 'Unknown',
            lat or '', lng or '',
            datetime.now().isoformat(), 'live_upload'
        ])
    print(f'✅ Saved to dataset: [{label}] {img_id}')
    return img_id


def predict_image(img_path, threshold=0.5):
    """Run model inference on a single image"""
    model.eval()
    img = Image.open(img_path).convert('RGB')
    tensor = eval_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1)[0].cpu().numpy()
    damaged_prob = float(probs[1])
    label = 'damaged' if damaged_prob >= threshold else 'clean'
    return label, damaged_prob, float(probs[0])


print('✅ Dataset manager and predictor ready')
print(f'   CSV location   : {DATASET_CSV}')
print(f'   Images folder  : {SAVED_IMAGES}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# UPLOAD YOUR ROAD PHOTO & TEST
# ─────────────────────────────────────────────────────────────────
from google.colab import files as colab_files

# ── Step 1: Upload image ──
print('📸 Upload a road photo:')
uploaded = colab_files.upload()
img_path = list(uploaded.keys())[0]

# ── Step 2: Enter location info ──
print('\n📍 Enter location info (press Enter to skip):')
loc_name = input('Location name (e.g. MG Road, Nagpur): ').strip() or 'Unknown'
lat_str  = input('Latitude  (e.g. 21.1458): ').strip()
lng_str  = input('Longitude (e.g. 79.0882): ').strip()
lat = float(lat_str) if lat_str else None
lng = float(lng_str) if lng_str else None

# ── Step 3: Run detection ──
label, dam_prob, clean_prob = predict_image(img_path)

# ── Step 4: Save to dataset automatically ──
entry_id = save_to_dataset(img_path, label, dam_prob, loc_name, lat, lng)

# ── Step 5: Visualize result ──
img_orig = Image.open(img_path).convert('RGB')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0a0c0f')

# Left: annotated image
ax = axes[0]
ax.imshow(img_orig)
ax.axis('off')
result_color = '#e84545' if label == 'damaged' else '#3ddc84'
result_text  = '⚠ ROAD DAMAGE DETECTED' if label == 'damaged' else '✅ CLEAN ROAD'
ax.set_title(result_text, color=result_color, fontsize=14, fontweight='bold', pad=10)
# Overlay box
box_props = dict(boxstyle='round,pad=0.4', facecolor=result_color, alpha=0.85)
ax.text(0.5, 0.05, f'Confidence: {max(dam_prob, clean_prob)*100:.1f}%',
        transform=ax.transAxes, ha='center',
        fontsize=12, color='white', fontweight='bold', bbox=box_props)

# Right: probability bar chart
ax2 = axes[1]
ax2.set_facecolor('#111418')
categories = ['Clean Road', 'Damaged Road']
values     = [clean_prob * 100, dam_prob * 100]
bar_colors = ['#3ddc84', '#e84545']
bars = ax2.barh(categories, values, color=bar_colors, height=0.4)
ax2.set_xlim(0, 110)
for bar, v in zip(bars, values):
    ax2.text(v+1, bar.get_y()+bar.get_height()/2,
             f'{v:.1f}%', va='center', color='#e8eaf0',
             fontsize=14, fontweight='bold')
ax2.axvline(50, color='#f5a623', linestyle='--', lw=1.5, label='Decision boundary (50%)')
ax2.set_xlabel('Probability (%)', color='#6b7280')
ax2.set_title('Detection Confidence', color='#f5a623', fontweight='bold')
ax2.tick_params(colors='#e8eaf0')
for sp in ax2.spines.values(): sp.set_color('#252b35')
ax2.legend(facecolor='#181c22', labelcolor='#e8eaf0')

# Info panel
info = f'ID: {entry_id}\nLocation: {loc_name}'
if lat: info += f'\nGPS: {lat:.4f}, {lng:.4f}'
info += f'\nTime: {datetime.now().strftime("%Y-%m-%d %H:%M")}'
ax2.text(0.98, 0.02, info, transform=ax2.transAxes, ha='right', va='bottom',
         color='#6b7280', fontsize=8, fontfamily='monospace')

fig.suptitle('🚧 RoadScan AI — Detection Result', color='#f5a623',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('detection_result.png', dpi=150, bbox_inches='tight', facecolor='#0a0c0f')
plt.show()

# Print summary
print(f'\n{"="*45}')
print(f'  RESULT   : {result_text}')
print(f'  P(damage): {dam_prob*100:.1f}%')
print(f'  P(clean) : {clean_prob*100:.1f}%')
print(f'  Location : {loc_name}')
print(f'  Saved as : {entry_id}')
print(f'{"="*45}')

---
## 📋 CELL 9 — View & Analyze Live Dataset

In [ ]:
# ── Load and display the live dataset ────────────────────────────
df = pd.read_csv(DATASET_CSV)
print(f'Total records in live dataset: {len(df)}')
print(f'Labels: {df["label"].value_counts().to_dict()}')
print()
print(df.tail(10).to_string(index=False))

if len(df) >= 4:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.patch.set_facecolor('#0a0c0f')
    for ax in axes:
        ax.set_facecolor('#111418')
        for sp in ax.spines.values(): sp.set_color('#252b35')
        ax.tick_params(colors='#6b7280')

    # Label distribution
    counts = df['label'].value_counts()
    axes[0].pie(counts.values, labels=counts.index,
                colors=['#3ddc84','#e84545'],
                autopct='%1.0f%%', textprops={'color':'#e8eaf0'})
    axes[0].set_title('Clean vs Damaged', color='#f5a623', fontweight='bold')

    # Damage probability distribution
    dam = df[df.label=='damaged']['damage_probability'].astype(float)
    cln = df[df.label=='clean']['damage_probability'].astype(float)
    if len(dam): axes[1].hist(dam, bins=10, color='#e84545', alpha=0.8, label='Damaged')
    if len(cln): axes[1].hist(cln, bins=10, color='#3ddc84', alpha=0.8, label='Clean')
    axes[1].axvline(0.5, color='#f5a623', linestyle='--')
    axes[1].set_title('P(damage) Distribution', color='#f5a623', fontweight='bold')
    axes[1].set_xlabel('Probability', color='#6b7280')
    axes[1].legend(facecolor='#181c22', labelcolor='#e8eaf0')

    # Confidence levels
    conf_counts = df['confidence_level'].value_counts()
    axes[2].bar(conf_counts.index, conf_counts.values,
                color=['#3ddc84','#f5a623','#e84545'][:len(conf_counts)])
    axes[2].set_title('Confidence Levels', color='#f5a623', fontweight='bold')
    axes[2].set_ylabel('Count', color='#6b7280')
    for ax in axes: ax.tick_params(colors='#e8eaf0')

    fig.suptitle('Live Dataset Analysis', color='#f5a623', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 💾 CELL 10 — Export & Download Everything

In [ ]:
import torch.onnx

# ── Export ONNX ───────────────────────────────────────────────────
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
torch.onnx.export(
    model, dummy, 'road_damage_model.onnx',
    export_params=True, opset_version=11,
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0:'batch'}, 'logits': {0:'batch'}}
)
print('✅ ONNX model saved')

# ── Save model metadata ───────────────────────────────────────────
metadata = {
    'model_arch'     : 'efficientnet_b4',
    'num_classes'    : 2,
    'classes'        : ['clean', 'damaged'],
    'threshold'      : 0.5,
    'input_size'     : IMG_SIZE,
    'normalize_mean' : [0.485, 0.456, 0.406],
    'normalize_std'  : [0.229, 0.224, 0.225],
    'test_accuracy'  : round(test_acc, 2),
    'test_f1'        : round(test_f1, 2),
    'roc_auc'        : round(float(roc_auc), 4),
    'trained_at'     : datetime.now().isoformat()
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('✅ Metadata saved')

# ── Package everything ────────────────────────────────────────────
import zipfile
with zipfile.ZipFile('roadscan_ai_export.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in [
        'road_damage_model.pth',
        'road_damage_model.onnx',
        'model_metadata.json',
        'road_damage_live_dataset.csv',
        'evaluation_report.png',
    ]:
        if Path(fname).exists():
            zf.write(fname)
            print(f'  Added: {fname}')
print('\n✅ Package: roadscan_ai_export.zip')

# ── Download ──────────────────────────────────────────────────────
print('\nDownloading files...')
for fname in ['roadscan_ai_export.zip', 'road_damage_live_dataset.csv', 'evaluation_report.png']:
    if Path(fname).exists():
        colab_files.download(fname)
        print(f'  ⬇ {fname}')

---
## 🌐 CELL 11 — Gradio Live Demo (Public URL)

In [ ]:
import gradio as gr

model.eval()

def gradio_predict(image, location_name, latitude, longitude):
    """Gradio interface function"""
    # Convert Gradio image (numpy array) to PIL
    pil_img = Image.fromarray(image.astype('uint8')).convert('RGB')

    # Save temp file
    tmp_path = '/tmp/gradio_input.jpg'
    pil_img.save(tmp_path)

    # Predict
    label, dam_prob, clean_prob = predict_image(tmp_path)

    # Auto-save to dataset
    lat = float(latitude) if str(latitude).strip() else None
    lng = float(longitude) if str(longitude).strip() else None
    entry_id = save_to_dataset(tmp_path, label, dam_prob,
                                location_name or 'Unknown', lat, lng)

    # Build result
    result_label = {
        'Clean Road ✅' : clean_prob,
        'Damaged Road ⚠️': dam_prob
    }

    status = ('⚠️ ROAD DAMAGE DETECTED — Report filed!'
               if label == 'damaged'
               else '✅ CLEAN ROAD — No damage detected')

    info = (
        f"Report ID : {entry_id}\n"
        f"Label     : {label.upper()}\n"
        f"Damage %  : {dam_prob*100:.1f}%\n"
        f"Clean %   : {clean_prob*100:.1f}%\n"
        f"Location  : {location_name or 'Not provided'}\n"
        f"Saved to dataset ✓"
    )
    return result_label, status, info


with gr.Blocks(theme=gr.themes.Base(), title='RoadScan AI') as demo:
    gr.Markdown("""# 🚧 RoadScan AI — Road Damage Detection
**Upload a road photo → AI detects damage → Auto-saved to dataset**
""")

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(label='Road Photo', type='numpy', height=280)
            loc_name  = gr.Textbox(label='Location Name', placeholder='e.g. MG Road, Nagpur')
            with gr.Row():
                lat_in = gr.Number(label='Latitude',  precision=6, value=None)
                lng_in = gr.Number(label='Longitude', precision=6, value=None)
            btn = gr.Button('🔍 Detect Road Damage', variant='primary', size='lg')

        with gr.Column(scale=1):
            label_out  = gr.Label(label='Detection Result', num_top_classes=2)
            status_out = gr.Textbox(label='Status', lines=1)
            info_out   = gr.Textbox(label='Report Details', lines=7)

    btn.click(
        fn=gradio_predict,
        inputs=[img_input, loc_name, lat_in, lng_in],
        outputs=[label_out, status_out, info_out]
    )

    gr.Examples(
        examples=[
            [str(p), 'Sample location', 21.1458, 79.0882]
            for p in list(Path('data/split/test/damaged').glob('*.jpg'))[:3]
        ],
        inputs=[img_input, loc_name, lat_in, lng_in]
    )

demo.launch(share=True, debug=False)
print('\n🌐 Public URL printed above — share with anyone!')